In [ ]:
# 安装必要依赖（Kaggle Notebook中可先执行）
# !pip install kagglehub d2l torchvision --upgrade

import d2l.torch as d2l
import numpy as np
import pandas as pd
import random
import time
import torch
import os
import kagglehub

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import models
from torchvision.transforms import v2

# ===================== 1. 配置超参数 =====================
TEST_SIZE = 0.1
BATCH_SIZE = 128
LR = 1e-4
L2 = 1e-3
EPOCHS = 50
# 随机种子（保证实验可复现）
SEED = 3407

# ===================== 2. 下载数据集并获取路径 =====================
def get_dataset_path():
    """下载数据集并返回根路径"""
    # 下载classify-leaves竞赛数据（自动缓存，重复运行不会重复下载）
    dataset_path = kagglehub.competition_download('classify-leaves')
    print(f"数据集下载到: {dataset_path}")
    
    # 验证关键文件是否存在
    train_csv_path = os.path.join(dataset_path, 'train.csv')
    test_csv_path = os.path.join(dataset_path, 'test.csv')
    images_dir = os.path.join(dataset_path)  # 图片文件直接在根目录
    
    if not os.path.exists(train_csv_path):
        raise FileNotFoundError(f"未找到train.csv，路径：{train_csv_path}")
    if not os.path.exists(test_csv_path):
        raise FileNotFoundError(f"未找到test.csv，路径：{test_csv_path}")
    
    return dataset_path, train_csv_path, test_csv_path, images_dir

# 执行下载并获取路径
dataset_root, train_csv, test_csv, images_dir = get_dataset_path()

# ===================== 3. 设置随机种子 =====================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)

# ===================== 4. 自定义数据集类 =====================
class LeaveDataset(Dataset):
    def __init__(self, img_names, labels, images_dir, train=True):
        self.img_names = img_names  # 图片文件名列表
        self.labels = labels        # 标签列表（训练集为数字，测试集可为任意值）
        self.images_dir = images_dir  # 图片根目录
        self.train = train          # 是否为训练集（控制数据增强）
        
        # 训练集数据增强（适配树叶分类的温和增强）
        self.train_trans = v2.Compose([
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomVerticalFlip(p=0.5),
            v2.RandomRotation(degrees=15, fill=(255, 255, 255)),
            v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            v2.Resize((224, 224)),
            v2.PILToTensor(),
            v2.ToDtype(torch.float32, True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        
        # 验证/测试集变换（无增强，仅必要预处理）
        self.eval_trans = v2.Compose([
            v2.Resize((224, 224)),
            v2.PILToTensor(),
            v2.ToDtype(torch.float32, True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    
    def __getitem__(self, index):
        # 读取图片
        img_name = self.img_names[index]
        img_path = os.path.join(self.images_dir, img_name)
        try:
            img = Image.open(img_path).convert('RGB')  # 强制转RGB（避免灰度图问题）
        except Exception as e:
            print(f"读取图片失败 {img_path}: {e}")
            raise e
        
        # 应用变换
        if self.train:
            img = self.train_trans(img)
        else:
            img = self.eval_trans(img)
        
        # 返回图片和标签（测试集标签无意义，仍返回）
        label = self.labels[index] if self.labels is not None else 0
        return img, label
    
    def __len__(self):
        return len(self.img_names)

# ===================== 5. 加载训练/验证数据 =====================
def load_train_valid_data(train_csv, images_dir):
    """加载训练数据并拆分训练/验证集"""
    # 读取训练CSV
    train_df = pd.read_csv(train_csv)
    print(f"训练数据总量: {len(train_df)}")
    
    # 标签编码（字符串→数字）
    le = LabelEncoder()
    train_df['label_encoded'] = le.fit_transform(train_df['label'])
    
    # 拆分训练/验证集（分层抽样，保证类别分布一致）
    train_imgs, valid_imgs, train_labels, valid_labels = train_test_split(
        train_df['image'].values,
        train_df['label_encoded'].values,
        test_size=TEST_SIZE,
        random_state=SEED,
        shuffle=True,
        stratify=train_df['label_encoded'].values
    )
    
    print(f"训练集数量: {len(train_imgs)}, 验证集数量: {len(valid_imgs)}")
    
    # 创建数据集
    train_dataset = LeaveDataset(train_imgs, train_labels, images_dir, train=True)
    valid_dataset = LeaveDataset(valid_imgs, valid_labels, images_dir, train=False)
    
    # 创建DataLoader
    # Kaggle中num_workers建议设为2（避免多线程报错），persistent_workers加速
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,  # 训练集必须shuffle
        num_workers=2,
        persistent_workers=True,
        pin_memory=True  # 加速GPU数据传输
    )
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        persistent_workers=True,
        pin_memory=True
    )
    
    print(f"训练Loader批次: {len(train_loader)}, 验证Loader批次: {len(valid_loader)}")
    return train_loader, valid_loader, le

# 执行数据加载
train_loader, valid_loader, label_encoder = load_train_valid_data(train_csv, images_dir)

# ===================== 6. 构建模型 =====================
def build_model(num_classes):
    """构建ResNet18迁移学习模型"""
    # 加载预训练ResNet18
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    
    # 替换最后一层（适配树叶分类）
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(0.5),  # Dropout概率0.5，防止过拟合
        nn.Linear(128, num_classes)
    )
    
    # 移到GPU/CPU
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    print(f"模型部署到设备: {device}")
    return model, device

# 构建模型（类别数=标签编码器的类别数）
model, device = build_model(len(label_encoder.classes_))

# ===================== 7. 配置优化器和损失函数 =====================
# 损失函数（分类任务用交叉熵）
criterion = nn.CrossEntropyLoss()

# 优化器（分层学习率：主干网络低学习率，分类头高学习率）
optimizer = torch.optim.AdamW(
    [
        # 主干网络参数（除fc外）
        {'params': [p for n, p in model.named_parameters() if 'fc' not in n], 'lr': LR},
        # 分类头参数（学习率×10）
        {'params': model.fc.parameters(), 'lr': LR * 10}
    ],
    weight_decay=L2  # L2正则
)

# 学习率调度器（余弦退火）
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    verbose=True  # 打印学习率变化
)

# ===================== 8. 训练和验证函数 =====================
def train_one_epoch(model, loader, criterion, optimizer, device, train_mode=True):
    """训练/验证一个epoch"""
    if train_mode:
        model.train()
        desc = "训练"
    else:
        model.eval()
        desc = "验证"
    
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    
    with torch.set_grad_enabled(train_mode):  # 验证集禁用梯度
        for batch_idx, (imgs, labels) in enumerate(loader):
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, dtype=torch.long, non_blocking=True)
            
            # 前向传播
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            
            # 训练集反向传播+更新
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            
            # 计算指标
            total_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            total_correct += torch.sum(preds == labels.data)
            total_samples += imgs.size(0)
            
            # 打印批次进度（可选）
            if batch_idx % 20 == 0:
                print(f"{desc}批次 {batch_idx}/{len(loader)}, 批次损失: {loss.item():.4f}")
    
    # 计算平均损失和准确率
    avg_loss = total_loss / total_samples
    avg_acc = total_correct.double() / total_samples
    return avg_loss, avg_acc

# ===================== 9. 主训练循环 =====================
def main_train_loop(model, train_loader, valid_loader, criterion, optimizer, scheduler, device, epochs):
    """主训练循环"""
    best_val_acc = 0.0  # 记录最佳验证集准确率
    best_model_path = "best_leaves_model.pth"  # 最佳模型保存路径
    
    for epoch in range(epochs):
        start_time = time.time()
        
        # 训练
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device, train_mode=True
        )
        
        # 验证
        val_loss, val_acc = train_one_epoch(
            model, valid_loader, criterion, optimizer, device, train_mode=False
        )
        
        # 更新学习率
        scheduler.step()
        
        # 保存最佳模型
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_acc': best_val_acc,
                'label_encoder': label_encoder  # 保存标签编码器
            }, best_model_path)
            print(f"保存最佳模型，验证准确率: {best_val_acc:.4f}")
        
        # 打印epoch结果
        epoch_time = time.time() - start_time
        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"训练损失: {train_loss:.6f}, 训练准确率: {train_acc:.6f}")
        print(f"验证损失: {val_loss:.6f}, 验证准确率: {val_acc:.6f}")
        print(f"耗时: {epoch_time:.2f}秒, 最佳验证准确率: {best_val_acc:.6f}")
        print("-" * 50)

# 启动训练
main_train_loop(model, train_loader, valid_loader, criterion, optimizer, scheduler, device, EPOCHS)

# ===================== 10. 测试集预测并生成提交文件 =====================
def predict_test_set(test_csv, images_dir, best_model_path, label_encoder):
    """对测试集预测并生成提交文件"""
    # 读取测试CSV
    test_df = pd.read_csv(test_csv)
    test_imgs = test_df['image'].values
    print(f"测试集数量: {len(test_imgs)}")
    
    # 创建测试数据集
    test_dataset = LeaveDataset(test_imgs, labels=None, images_dir=images_dir, train=False)
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    
    # 加载最佳模型
    checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # 预测
    all_preds = []
    with torch.no_grad():
        for imgs, _ in test_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            # 数字标签转回字符串
            preds_str = label_encoder.inverse_transform(preds)
            all_preds.extend(preds_str)
    
    # 生成提交文件
    test_df['label'] = all_preds
    submission_path = "submission.csv"
    test_df[['image', 'label']].to_csv(submission_path, index=False)
    print(f"提交文件已保存到: {submission_path}")
    return test_df

# 执行测试集预测
predict_test_set(test_csv, images_dir, "best_leaves_model.pth", label_encoder)
#kaggle上交
kaggle competitions submit -c classify-leaves -f submission.csv -k fengdongplus1/<NOTEBOOK> -v <VERSION> -m "Message"